# load Data

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)
from sklearn.linear_model import LinearRegression

In [3]:
df = pd.read_csv(r"Prod_data\tbl_device_history.csv")

In [12]:
df.head()

,historyId,deviceEndpoint,latitude,longitude,trolleyId,geozoneId,geolayerId,createdOn,modifiedOn,status,nestId,wifiRTT,locationSource,address
0,1,TR1-B43A4535C59C,2.754915,101.703118,1,11,7,04-03-2026 13:35,NaN,1,0,[SSID:BC1-F412FA8990B9 BSSID:F4:12:FA:89:90:B9...,Wi-Fi,"Short Term Car Park 'B', Jalan CTA 2, Sepang, ..."
1,2,TR1-B43A4535C55C,2.754909,101.703125,2,11,7,04-03-2026 13:35,NaN,1,0,[SSID:BC1-F412FA8990B9 BSSID:F4:12:FA:89:90:B9...,Wi-Fi,"Short Term Car Park 'B', Jalan CTA 2, Sepang, ..."
2,3,TR1-B43A4535C59C,2.754895,101.703052,1,11,7,04-03-2026 13:37,NaN,1,0,[SSID:BC1-F412FA8990B9 BSSID:F4:12:FA:89:90:B9...,Wi-Fi,"Short Term Car Park 'B', Jalan CTA 2, Sepang, ..."
3,4,TR1-B43A4535C55C,2.754914,101.703067,2,11,7,04-03-2026 13:38,NaN,1,0,[SSID:BC1-F412FA8990B9 BSSID:F4:12:FA:89:90:B9...,Wi-Fi,"Short Term Car Park 'B', Jalan CTA 2, Sepang, ..."
4,5,TR1-B43A4535C338,2.754914,101.703097,7,11,7,04-03-2026 13:38,04-03-2026 14:06,1,0,[SSID:BC1-F412FA8990B9 BSSID:F4:12:FA:89:90:B9...,Wi-Fi,"Short Term Car Park 'B', Jalan CTA 2, Sepang, ..."


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 299120 entries, 0 to 299119
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   historyId       299120 non-null  int64  
 1   deviceEndpoint  299120 non-null  str    
 2   latitude        299120 non-null  float64
 3   longitude       299120 non-null  float64
 4   trolleyId       299120 non-null  int64  
 5   geozoneId       299120 non-null  int64  
 6   geolayerId      299120 non-null  int64  
 7   createdOn       299120 non-null  str    
 8   modifiedOn      26011 non-null   str    
 9   status          299120 non-null  int64  
 10  nestId          299120 non-null  int64  
 11  wifiRTT         299120 non-null  str    
 12  locationSource  299120 non-null  str    
 13  address         166210 non-null  str    
dtypes: float64(2), int64(6), str(6)
memory usage: 105.5 MB


# Cell 3 - Preprocessing

In [13]:
def build_trolley_count(raw):
    
    df = raw.copy()

    # ------------------------------------------------------------------
    # Datetime Processing
    # ------------------------------------------------------------------
    df["createdOn"] = pd.to_datetime(
        df["createdOn"],
        format="%d-%m-%Y %H:%M"
    )

    df["date"] = df["createdOn"].dt.date
    df["time"] = df["createdOn"].dt.time

    df.drop(columns=["createdOn"], inplace=True)

    df["datetime"] = pd.to_datetime(
        df["date"].astype(str) + " " + df["time"].astype(str)
    )

    df = (
        df.sort_values(
            ["trolleyId", "datetime"]
        )
        .reset_index(drop=True)
    )

    # ------------------------------------------------------------------
    # Stay Detection
    # ------------------------------------------------------------------
    df["zone_changed"] = (
        (df["trolleyId"] != df["trolleyId"].shift(1))
        |
        (df["geozoneId"] != df["geozoneId"].shift(1))
    )

    df["stay_id"] = df["zone_changed"].cumsum()

    stays = (
        df.groupby("stay_id")
        .agg(
            trolleyId=("trolleyId", "first"),
            geozoneId=("geozoneId", "first"),
            geolayerId=("geolayerId", "first"),
            enter_time=("datetime", "min"),
            exit_time=("datetime", "max"),
            ping_count=("historyId", "count")
        )
        .reset_index(drop=True)
    )

    stays["duration_mins"] = (
        (
            stays["exit_time"] - stays["enter_time"]
        )
        .dt.total_seconds()
        / 60
    )

    stays["is_long_stay"] = (
        stays["duration_mins"] >= 120
    ).astype(int)

    stays = stays[
        stays["geozoneId"] != 0
    ].copy()

    # ------------------------------------------------------------------
    # Hour Expansion
    # ------------------------------------------------------------------
    min_slot = stays["enter_time"].min().floor("h")
    max_slot = stays["exit_time"].max().ceil("h")

    all_slots = pd.date_range(
        start=min_slot,
        end=max_slot,
        freq="1h"
    )

    records = []

    for _, row in stays.iterrows():

        for slot_start in all_slots:

            slot_end = slot_start + pd.Timedelta(hours=1)

            if (
                row["enter_time"] < slot_end
                and
                row["exit_time"] > slot_start
            ):
                records.append({
                    "trolleyId": row["trolleyId"],
                    "geozoneId": row["geozoneId"],
                    "geolayerId": row["geolayerId"],
                    "slot_start": slot_start,
                    "date": slot_start.date(),
                    "hour": slot_start.hour,
                    "day_of_week": slot_start.day_name(),
                    "day_num": slot_start.dayofweek,
                    "is_weekend": int(slot_start.dayofweek >= 5)
                })

    expanded = pd.DataFrame(records)

    trolley_count = (
        expanded.groupby([
            "date",
            "day_of_week",
            "day_num",
            "is_weekend",
            "hour",
            "slot_start",
            "geozoneId",
            "geolayerId"
        ])["trolleyId"]
        .nunique()
        .reset_index()
        .rename(columns={"trolleyId": "trolley_count"})
    )

    trolley_count["date"] = pd.to_datetime(
        trolley_count["date"]
    )

    trolley_count["month"] = (
        trolley_count["date"].dt.month
    )

    return trolley_count

In [14]:
def preprocess(df):

    df = df.copy()

    # ---------------------------------------------------
    # Null Handling
    # ---------------------------------------------------
    for col in df.columns:

        null_pct = df[col].isnull().mean()

        if 0 < null_pct < 0.30:
            df.dropna(subset=[col], inplace=True)

    # ---------------------------------------------------
    # Cyclical Features
    # ---------------------------------------------------

    df["hour_sin"] = np.sin(
        2 * np.pi * df["hour"] / 24
    )

    df["hour_cos"] = np.cos(
        2 * np.pi * df["hour"] / 24
    )

    df["month_sin"] = np.sin(
        2 * np.pi * df["month"] / 12
    )

    df["month_cos"] = np.cos(
        2 * np.pi * df["month"] / 12
    )

    df["day_sin"] = np.sin(
        2 * np.pi * df["day_num"] / 7
    )

    df["day_cos"] = np.cos(
        2 * np.pi * df["day_num"] / 7
    )

    return df

In [15]:
raw = pd.read_csv(r"Prod_data\tbl_device_history.csv")

tc_df = build_trolley_count(raw)

df = preprocess(tc_df)

FEATURES = [
    "day_num",
    "is_weekend",
    "hour",
    "geozoneId",
    "geolayerId",
    "month",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "day_sin",
    "day_cos"
]

TARGET = "trolley_count"

X = df[FEATURES]
y = df[TARGET]

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [17]:
models = {
    "DecisionTree": DecisionTreeRegressor(
        random_state=42
    ),

    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "GradientBoosting": GradientBoostingRegressor(
        random_state=42
    ),

    "AdaBoost": AdaBoostRegressor(
        random_state=42
    )
}

In [18]:
results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            pred
        )
    )

    r2 = r2_score(
        y_test,
        pred
    )

    results.append({
        "Model": name,
        "MAE": round(mae, 3),
        "RMSE": round(rmse, 3),
        "R2": round(r2, 4)
    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    "R2",
    ascending=False
)

,Model,MAE,RMSE,R2
1,RandomForest,6.957,17.562,0.7271
3,GradientBoosting,8.255,17.644,0.7246
2,ExtraTrees,7.072,18.308,0.7035
0,DecisionTree,7.455,18.942,0.6826
4,AdaBoost,12.564,22.764,0.5416


In [19]:
# ============================================================
# TRAIN RANDOM FOREST
# ============================================================

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

In [20]:
rf_model.fit(X_train, y_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [21]:
# ============================================================
# PREDICTIONS
# ============================================================

y_pred = rf_model.predict(X_test)

In [22]:
# ============================================================
# EVALUATION
# ============================================================

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

print("Random Forest Results")
print("-" * 40)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

Random Forest Results
----------------------------------------
MAE  : 6.9569
RMSE : 17.5621
R²   : 0.7271


In [23]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

feature_importance = (
    pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": rf_model.feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

print("\nFeature Importance")
display(feature_importance)


Feature Importance


,Feature,Importance
4,geolayerId,0.433631
3,geozoneId,0.190356
8,month_sin,0.073085
5,month,0.070087
9,month_cos,0.061459
10,day_sin,0.038907
2,hour,0.033022
6,hour_sin,0.027744
11,day_cos,0.024455
7,hour_cos,0.023652


In [25]:
joblib.dump(
    rf_model,
    "Stracker_randomforest_model_demand_forest.pkl"
)

['Stracker_randomforest_model_demand_forest.pkl']

In [27]:
pip uninstall scikit-learn

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
aa=joblib.load(r'Stracker_randomforest_model_demand_forest.pkl')